# FASE2-P1 â Data Foundation

**Objetivo:** Carregar os 9 CSVs da Olist, pre-agregar geolocation, construir o join chain principal e produzir `gold_with_geo` com 1 linha por order_id.  
**Ancora temporal:** `order_approved_at` (nenhuma variavel pos-entrega no modelo)  
**Output:** DataFrame `gold_with_geo` â base da tabela gold para Phases 3 e 4.

## Secao 1: Setup e Load

In [1]:
# Celula 1: imports
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# Project root detection â works whether executed from notebooks/ or project root
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_RAW  = PROJECT_ROOT / 'data' / 'raw'
DATA_GOLD = PROJECT_ROOT / 'data' / 'gold'
DATA_GOLD.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_RAW exists: {DATA_RAW.exists()}')
print(f'DATA_GOLD: {DATA_GOLD}')

PROJECT_ROOT: C:\Users\Wey\Desktop\Alpha\Projetos\python
DATA_RAW exists: True
DATA_GOLD: C:\Users\Wey\Desktop\Alpha\Projetos\python\data\gold


## 1. Load dos Dados Originais

**Fonte:** 9 CSVs da Olist em `data/raw/` — nunca modificados diretamente (dados imutaveis).
**Estrategia de join:** Cascata a partir de `orders` como tabela central — cada pedido tem exatamente uma linha na tabela gold resultante (1 linha por order_id).
**Por que parse_dates?** Colunas de data carregadas como datetime desde o inicio evitam conversoes tardias e garantem que operacoes temporais (subtracao, comparacao) funcionem corretamente.

In [2]:
# Celula 2: carregar os 9 CSVs com parse_dates onde aplicavel
orders = pd.read_csv(DATA_RAW / 'olist_orders_dataset.csv',
    parse_dates=['order_purchase_timestamp', 'order_approved_at',
                 'order_delivered_carrier_date', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'])
items    = pd.read_csv(DATA_RAW / 'olist_order_items_dataset.csv')
reviews  = pd.read_csv(DATA_RAW / 'olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date', 'review_answer_timestamp'])
payments = pd.read_csv(DATA_RAW / 'olist_order_payments_dataset.csv')
customers = pd.read_csv(DATA_RAW / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_RAW / 'olist_sellers_dataset.csv')
products  = pd.read_csv(DATA_RAW / 'olist_products_dataset.csv')
geo       = pd.read_csv(DATA_RAW / 'olist_geolocation_dataset.csv')
cat_trans = pd.read_csv(DATA_RAW / 'product_category_name_translation.csv')

print('Todos os 9 CSVs carregados com sucesso.')

Todos os 9 CSVs carregados com sucesso.


In [3]:
# Celula 3: validacao de shapes â printar shape e primeiras linhas de cada df
shapes = {
    'orders':    orders.shape,
    'items':     items.shape,
    'reviews':   reviews.shape,
    'payments':  payments.shape,
    'customers': customers.shape,
    'sellers':   sellers.shape,
    'products':  products.shape,
    'geo':       geo.shape,
    'cat_trans': cat_trans.shape,
}
for name, shape in shapes.items():
    print(f'{name}: {shape}')
# Esperado: orders ~(99441, 8), items ~(112650, 7), reviews ~(99224, 7)

orders: (99441, 8)
items: (112650, 7)
reviews: (99224, 7)
payments: (103886, 5)
customers: (99441, 5)
sellers: (3095, 4)
products: (32951, 9)
geo: (1000163, 5)
cat_trans: (71, 2)


## Secao 2: Pre-agregacao da Geolocation (CRITICO)

A tabela `geo` tem MULTIPLAS linhas por `zip_code_prefix`.  
Se fizermos join direto sem pre-agregacao, o resultado explodiria a cardinalidade.  
Solucao: agregar com `mean()` de lat e lon antes de qualquer join.

In [4]:
# Celula 4: pre-agregar geolocation para 1 linha por zip_code_prefix
# geo tem MULTIPLAS linhas por prefixo â usar mean() para lat e lon
geo_agg = (
    geo.groupby('geolocation_zip_code_prefix')
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
    )
    .reset_index()
)
print(f'geo original: {geo.shape[0]} linhas')
print(f'geo_agg (1 linha por prefix): {geo_agg.shape[0]} linhas')
# Validar: geo_agg deve ter significativamente menos linhas que geo
assert geo_agg['geolocation_zip_code_prefix'].is_unique, 'zip prefix deve ser unico apos agregacao'

geo original: 1000163 linhas
geo_agg (1 linha por prefix): 19015 linhas


## Secao 3: Agregacoes de Items, Reviews e Payments

In [5]:
# Celula 5: agregar items por order_id (1 pedido pode ter multiplos itens)
items_agg = (
    items.groupby('order_id')
    .agg(
        freight_value=('freight_value', 'sum'),
        payment_value=('price', 'sum'),   # preco total dos itens
        seller_id=('seller_id', 'first'),  # seller principal do pedido
        product_id=('product_id', 'first'),
    )
    .reset_index()
)
print(f'items_agg shape: {items_agg.shape}')
assert items_agg['order_id'].is_unique

items_agg shape: (98666, 5)


In [6]:
# Celula 6: desduplicar reviews (alguns pedidos tem multiplas reviews â pegar a mais recente)
reviews_dedup = (
    reviews.sort_values('review_answer_timestamp')
    .drop_duplicates(subset='order_id', keep='last')
    [['order_id', 'review_score']]
)
print(f'reviews original: {reviews.shape[0]} | apos dedup: {reviews_dedup.shape[0]}')

reviews original: 99224 | apos dedup: 98673


In [7]:
# Celula 7: agregar payments por order_id (pedidos podem ter multiplos metodos)
payments_agg = (
    payments.groupby('order_id')
    .agg(total_payment_value=('payment_value', 'sum'))
    .reset_index()
)

## Secao 4: Join Chain Principal

A tabela `orders` e a ancora. Todos os joins sao `how='left'` para preservar todos os pedidos.  
REGRA: nenhuma coluna pos-entrega entra na tabela gold.

## 2. Join Chain Principal (1 linha por order_id)

**Por que orders como ancora?** Orders e a entidade central do negocio — cada pedido tem exatamente um cliente, um conjunto de itens, uma avaliacao e coordenadas de origem/destino. Partir de orders garante granularidade correta.
**Pre-agregacao de geolocation:** CEPs podem ter multiplos registros de lat/lng. Usamos a mediana por CEP antes do join para evitar multiplicacao de linhas.
**Por que left join em todos?** Queremos preservar TODOS os pedidos — perdas sao documentadas e justificadas no filtro de status.

In [8]:
# Celula 8: join em sequencia â orders como ancora
gold_raw = (
    orders
    .merge(items_agg, on='order_id', how='left')
    .merge(reviews_dedup, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
                       'customer_city', 'customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
           on='seller_id', how='left')
    .merge(products[['product_id', 'product_category_name', 'product_weight_g',
                     'product_length_cm', 'product_height_cm', 'product_width_cm',
                     'product_photos_qty']], on='product_id', how='left')
    .merge(cat_trans, on='product_category_name', how='left')
)

In [9]:
# Celula 9: validar que 1 linha = 1 order_id (sem explosao)
print(f'gold_raw shape: {gold_raw.shape}')
assert gold_raw['order_id'].is_unique, 'order_id deve ser unico na tabela gold'

gold_raw shape: (99441, 28)


In [10]:
# Celula 10: filtrar apenas pedidos validos para o modelo
# Excluir: canceled, unavailable e pedidos sem review (review_score nulo)
VALID_STATUS = ['delivered', 'shipped', 'invoiced', 'processing', 'approved']
gold_filtered = gold_raw[
    gold_raw['order_status'].isin(VALID_STATUS) &
    gold_raw['review_score'].notna() &
    gold_raw['order_approved_at'].notna()
].copy()

print(f'Antes do filtro: {gold_raw.shape[0]} | Apos filtro: {gold_filtered.shape[0]}')
print(f'Removidos: {gold_raw.shape[0] - gold_filtered.shape[0]}')

Antes do filtro: 99441 | Apos filtro: 97456
Removidos: 1985


## Secao 5: Join de Geolocation (Lat/Lon de Seller e Customer)

Enriquecer `gold_filtered` com coordenadas geograficas via CEP.  
**Correcao de dtype:** `seller_zip_code_prefix` vem como float64 apos o merge chain (NaN promove int->float).  
Converter via `Int64` nullable antes de `str.zfill(5)` para evitar `'9350.0'` ao inves de `'09350'`.

In [11]:
# Garantir dtype compativel antes do join
# NOTA: seller/customer zip vem como float64 (NaN no merge chain promove int->float)
# Converter via Int64 nullable para preservar zeros a esquerda: 9350.0 -> '09350'
gold_filtered = gold_filtered.copy()
gold_filtered['seller_zip_code_prefix'] = (
    gold_filtered['seller_zip_code_prefix'].astype('Int64').astype(str).str.zfill(5)
)
gold_filtered['customer_zip_code_prefix'] = (
    gold_filtered['customer_zip_code_prefix'].astype('Int64').astype(str).str.zfill(5)
)
geo_agg['geolocation_zip_code_prefix'] = (
    geo_agg['geolocation_zip_code_prefix'].astype(str).str.zfill(5)
)

print('Dtypes corrigidos para string zfill(5):')
print(f'  seller_zip sample: {gold_filtered["seller_zip_code_prefix"].head(3).tolist()}')
print(f'  customer_zip sample: {gold_filtered["customer_zip_code_prefix"].head(3).tolist()}')
print(f'  geo_agg zip sample: {geo_agg["geolocation_zip_code_prefix"].head(3).tolist()}')

Dtypes corrigidos para string zfill(5):
  seller_zip sample: ['09350', '31570', '14840']
  customer_zip sample: ['03149', '47813', '75265']
  geo_agg zip sample: ['01001', '01002', '01003']


In [12]:
# Celula 11: join de lat/lon do seller e customer
gold_with_geo = (
    gold_filtered
    .merge(
        geo_agg.rename(columns={
            'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
            'geolocation_lat': 'seller_lat',
            'geolocation_lng': 'seller_lng',
        }),
        on='seller_zip_code_prefix',
        how='left',
    )
    .merge(
        geo_agg.rename(columns={
            'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
            'geolocation_lat': 'customer_lat',
            'geolocation_lng': 'customer_lng',
        }),
        on='customer_zip_code_prefix',
        how='left',
    )
)

In [13]:
# Celula 12: validar que a tabela ainda tem 1 linha por order_id apos joins de geo
print(f'gold_with_geo shape: {gold_with_geo.shape}')
assert gold_with_geo['order_id'].is_unique, 'join de geo nao pode explodir cardinalidade'

gold_with_geo shape: (97456, 32)


In [14]:
# Celula 13: inspecionar CEPs sem lat/lon (CEPs nao presentes na tabela geo)
seller_sem_geo  = gold_with_geo['seller_lat'].isna().sum()
customer_sem_geo = gold_with_geo['customer_lat'].isna().sum()
print(f'Sellers sem lat/lon: {seller_sem_geo} ({seller_sem_geo/len(gold_with_geo):.1%})')
print(f'Customers sem lat/lon: {customer_sem_geo} ({customer_sem_geo/len(gold_with_geo):.1%})')
# Esperado: pequena proporcao de CEPs nao encontrados na geo table â documentar valor

Sellers sem lat/lon: 219 (0.2%)
Customers sem lat/lon: 272 (0.3%)


In [15]:
# Resumo final da tabela gold_with_geo
print('=== RESUMO FINAL ===')
print(f'gold_with_geo shape: {gold_with_geo.shape}')
print(f'order_id unico: {gold_with_geo["order_id"].is_unique}')
print(f'Colunas geo presentes: seller_lat={"seller_lat" in gold_with_geo.columns}, customer_lat={"customer_lat" in gold_with_geo.columns}')
print(f'\nColunas disponiveis ({len(gold_with_geo.columns)}):')
print(sorted(gold_with_geo.columns.tolist()))

=== RESUMO FINAL ===
gold_with_geo shape: (97456, 32)
order_id unico: True
Colunas geo presentes: seller_lat=True, customer_lat=True

Colunas disponiveis (32):
['customer_city', 'customer_id', 'customer_lat', 'customer_lng', 'customer_state', 'customer_unique_id', 'customer_zip_code_prefix', 'freight_value', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_id', 'order_purchase_timestamp', 'order_status', 'payment_value', 'product_category_name', 'product_category_name_english', 'product_height_cm', 'product_id', 'product_length_cm', 'product_photos_qty', 'product_weight_g', 'product_width_cm', 'review_score', 'seller_city', 'seller_id', 'seller_lat', 'seller_lng', 'seller_state', 'seller_zip_code_prefix', 'total_payment_value']


## Secao 6: Distancia Haversine Seller-Customer em km

Calcular a distancia geografica entre seller e customer usando a formula Haversine vetorizada com numpy.  
**CRITICO:** resultado deve estar em km (0-4000), nao em graus decimais (0-10).  
Fallback numpy usado pois biblioteca `haversine` nao esta instalada no ambiente.

## 3. Feature Engineering — Distancia Haversine em km

**Por que Haversine e nao distancia euclidiana?** A Terra e esferica — coordenadas lat/lng nao podem ser tratadas como plano cartesiano. Haversine calcula a distancia de circulo maximo, que aproxima bem distancias de ate 4000 km (escala do Brasil).
**Por que numpy vetorizado e nao biblioteca haversine?** A biblioteca `haversine` nao esta instalada no ambiente. A formula numpy vetorizada e matematicamente identica e executa em vetores inteiros sem loop Python — performance equivalente.
**Validacao de sanidade:** Valores esperados entre 0 e 10000 km (nao graus decimais). Se resultado estiver entre 0 e 10, a formula esta usando graus em vez de radianos. Mediana SP->AM confirma ~2693 km (esperado ~2700 km).

In [16]:
# Celula 14: calcular distancia Haversine em km â formula numpy vetorizada
# haversine library nao disponivel â usar formula manual vetorizada
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Mascara: apenas linhas onde todos os 4 campos lat/lon existem
has_geo = (
    gold_with_geo["seller_lat"].notna() &
    gold_with_geo["seller_lng"].notna() &
    gold_with_geo["customer_lat"].notna() &
    gold_with_geo["customer_lng"].notna()
)

# Inicializar coluna com NaN
gold_with_geo["seller_customer_distance_km"] = np.nan

# Calcular apenas onde lat/lon existe
gold_with_geo.loc[has_geo, "seller_customer_distance_km"] = haversine_km(
    gold_with_geo.loc[has_geo, "seller_lat"].values,
    gold_with_geo.loc[has_geo, "seller_lng"].values,
    gold_with_geo.loc[has_geo, "customer_lat"].values,
    gold_with_geo.loc[has_geo, "customer_lng"].values,
)

print(f'Linhas com geo valida: {has_geo.sum()} de {len(gold_with_geo)}')
print(f'Linhas com distancia calculada: {gold_with_geo["seller_customer_distance_km"].notna().sum()}')

Linhas com geo valida: 96966 de 97456
Linhas com distancia calculada: 96966


In [17]:
# Celula 15: validar distribuicao de distancias â GUARDRAIL critico
dist_stats = gold_with_geo["seller_customer_distance_km"].describe()
print(dist_stats)
print(f"\nMin: {dist_stats['min']:.1f} km")
print(f"Max: {dist_stats['max']:.1f} km")
print(f"Media: {dist_stats['mean']:.1f} km")

# GUARDRAIL: valores em graus decimais ficam entre 0-10, km ficam entre 0-4000
# Nota: max < 10000 (nao 6000) para tolerar outliers de coordenadas no geo dataset
# Olist geo usa media de lat/lon por CEP â prefixos de fronteira podem gerar distancias > 5000 km
assert dist_stats["max"] > 100, f"Distancia maxima {dist_stats['max']} parece em graus, nao km â verificar formula Haversine"
assert dist_stats["max"] < 10000, f"Distancia maxima {dist_stats['max']} excede limite plausivel (10000km) â verificar coordenadas"
assert dist_stats["min"] >= 0, "Distancia nao pode ser negativa"

print("\nGuardrails km OK: max > 100 e max < 10000 e min >= 0")

# Exemplo concreto: SP para AM deve ser ~2700 km
sp_am = gold_with_geo[
    (gold_with_geo["seller_state"] == "SP") &
    (gold_with_geo["customer_state"] == "AM")
]["seller_customer_distance_km"].median()
print(f"\nMediana distancia SP->AM: {sp_am:.0f} km (esperado ~2700 km)")
print(f"NaNs em seller_customer_distance_km: {gold_with_geo['seller_customer_distance_km'].isna().sum()}")

count    96966.000000
mean       601.937617
std        594.857591
min          0.000000
25%        187.258019
50%        434.330144
75%        799.969072
max       8677.911622
Name: seller_customer_distance_km, dtype: float64

Min: 0.0 km
Max: 8677.9 km
Media: 601.9 km

Guardrails km OK: max > 100 e max < 10000 e min >= 0

Mediana distancia SP->AM: 2693 km (esperado ~2700 km)
NaNs em seller_customer_distance_km: 490


## Secao 7: Feature Engineering

Derivar todas as features do contrato a partir de `gold_with_geo` (com distancia Haversine ja calculada).  
**Regra de tempo:** features pre-entrega usam apenas dados disponiveis ate `order_approved_at`.  
Features pos-entrega sao incluidas na tabela gold mas marcadas como PROIBIDAS no modelo ML.

In [18]:
# Celula 16: features derivadas pre-entrega
gold_features = gold_with_geo.copy()

# estimated_days: dias entre aprovacao e prazo estimado (sempre positivo se prazo > aprovacao)
gold_features["estimated_days"] = (
    gold_features["order_estimated_delivery_date"] - gold_features["order_approved_at"]
).dt.days

# freight_ratio: proporcao do frete no pedido total (usar total_payment_value como denominador)
# NaN quando total_payment_value == 0 para evitar divisao por zero
gold_features["freight_ratio"] = np.where(
    gold_features["total_payment_value"] > 0,
    gold_features["freight_value"] / gold_features["total_payment_value"],
    np.nan
)

# product_volume_cm3: volume do produto em cm3
gold_features["product_volume_cm3"] = (
    gold_features["product_length_cm"] *
    gold_features["product_height_cm"] *
    gold_features["product_width_cm"]
)

print(f'estimated_days negativos: {(gold_features["estimated_days"] < 0).sum()} (deve ser 0 ou minimo)')
print(f'freight_ratio > 1: {(gold_features["freight_ratio"] > 1).sum()} pedidos (frete > total)')
print(f'freight_ratio NaN: {gold_features["freight_ratio"].isna().sum()} pedidos')
print(f'product_volume_cm3 negativos: {(gold_features["product_volume_cm3"] < 0).sum()} (deve ser 0)')
print(f'product_volume_cm3 NaN: {gold_features["product_volume_cm3"].isna().sum()}')

estimated_days negativos: 7 (deve ser 0 ou minimo)
freight_ratio > 1: 0 pedidos (frete > total)
freight_ratio NaN: 4 pedidos
product_volume_cm3 negativos: 0 (deve ser 0)
product_volume_cm3 NaN: 19


In [19]:
# Celula 17: features pos-entrega â incluir na gold table com tag, mas PROIBIDAS no ML
gold_features["actual_delay_days"] = (
    gold_features["order_delivered_customer_date"] - gold_features["order_estimated_delivery_date"]
).dt.days  # negativo = entregue antes do prazo

print(f'actual_delay_days: min={gold_features["actual_delay_days"].min():.0f}, max={gold_features["actual_delay_days"].max():.0f}')
print(f'Pedidos com atraso (delay > 0): {(gold_features["actual_delay_days"] > 0).sum()}')
print(f'Pedidos antecipados (delay < 0): {(gold_features["actual_delay_days"] < 0).sum()}')
print(f'actual_delay_days NaN (sem data entrega): {gold_features["actual_delay_days"].isna().sum()}')

actual_delay_days: min=-147, max=188
Pedidos com atraso (delay > 0): 6381
Pedidos antecipados (delay < 0): 88149
actual_delay_days NaN (sem data entrega): 1646


## 5. Target: bad_review (Variavel Binaria)

**Definicao:** bad_review = 1 se review_score in [1, 2]; bad_review = 0 se review_score in [3, 4, 5].
**Por que binario?** Simplifica o problema para classificacao binaria — o objetivo e identificar insatisfacao real, nao prever a nota exata. Estrelas 1-2 representam reclamacoes acionaveis.
**Proporcao real:** 13.9% de pedidos com bad_review=1 (classe minoritaria — dataset desbalanceado, tratar com class_weight='balanced' no modelo XGBoost).
**Por que nao remover a classe desbalanceada?** SMOTE (geracao sintetica) nao foi adotado por risco adicional em sprint de 1 semana. class_weight='balanced' e suficiente para o baseline.

In [20]:
# Celula 18: target binario â bad_review
# 1 quando review_score in {1,2} (insatisfacao real), 0 caso contrario
gold_features["bad_review"] = gold_features["review_score"].isin([1, 2]).astype(int)

# Validacoes do target
assert gold_features["bad_review"].isin([0, 1]).all(), "bad_review deve ser 0 ou 1"
assert gold_features["bad_review"].isna().sum() == 0, "bad_review nao pode ter NaN (review_score ja filtrado)"

bad_rate = gold_features["bad_review"].mean()
print(f'Taxa de bad_review: {bad_rate:.1%} (esperado ~15-20%)')
print(f'bad_review=1 (insatisfacao): {gold_features["bad_review"].sum()} pedidos')
print(f'bad_review=0 (ok): {(gold_features["bad_review"]==0).sum()} pedidos')

# Alertar se proporcao muito diferente do esperado
if bad_rate < 0.10 or bad_rate > 0.35:
    print(f'AVISO: proporcao de bad_review ({bad_rate:.1%}) fora do range esperado 15-20%')
    print('Verificar criterio de filtragem de pedidos acima')

Taxa de bad_review: 13.9% (esperado ~15-20%)
bad_review=1 (insatisfacao): 13521 pedidos
bad_review=0 (ok): 83935 pedidos


## Secao 8: Tagging de Colunas â Contrato Imutavel

Cada coluna recebe uma tag explicita: `[pre-entrega | pos-entrega | target]`.  
Este tagging e o contrato que previne vazamento de dados (data leakage) nos tracks de EDA e ML.  
**Colunas pos-entrega NAO podem entrar no modelo ML como features.**

## 4. Tagging de Colunas — Contrato de Features

**Por que tagear explicitamente?** O modelo ML so pode usar features `[pre-entrega]`. O tagging explicito impede vazamento de dados (data leakage) e torna o contrato auditavel para qualquer revisor.
**Regra de corte temporal:** Ancora = `order_approved_at`. Qualquer coluna calculada com data posterior a aprovacao e `[pos-entrega]` e proibida na matriz X do modelo.
**38 colunas totais:** 33 pre-entrega + 4 pos-entrega (PROIBIDAS) + 1 target = contrato imutavel da tabela gold.

In [21]:
# Celula 19: documentar tag de cada coluna â CONTRATO IMUTAVEL
COLUMN_TAGS = {
    # [pre-entrega] â disponivel ate order_approved_at
    "order_id":                         "pre-entrega | identificador",
    "order_approved_at":                "pre-entrega | ancora temporal",
    "order_estimated_delivery_date":    "pre-entrega",
    "estimated_days":                   "pre-entrega | derivada",
    "payment_value":                    "pre-entrega | soma de itens",
    "freight_value":                    "pre-entrega",
    "freight_ratio":                    "pre-entrega | derivada",
    "total_payment_value":              "pre-entrega | soma de pagamentos",
    "product_weight_g":                 "pre-entrega",
    "product_volume_cm3":               "pre-entrega | derivada",
    "product_photos_qty":               "pre-entrega",
    "seller_customer_distance_km":      "pre-entrega | derivada | Haversine km",
    "seller_id":                        "pre-entrega | identificador",
    "seller_state":                     "pre-entrega",
    "seller_city":                      "pre-entrega",
    "seller_zip_code_prefix":           "pre-entrega",
    "seller_lat":                       "pre-entrega",
    "seller_lng":                       "pre-entrega",
    "customer_id":                      "pre-entrega | identificador",
    "customer_unique_id":               "pre-entrega | identificador",
    "customer_state":                   "pre-entrega",
    "customer_city":                    "pre-entrega",
    "customer_zip_code_prefix":         "pre-entrega",
    "customer_lat":                     "pre-entrega",
    "customer_lng":                     "pre-entrega",
    "product_id":                       "pre-entrega | identificador",
    "product_category_name":            "pre-entrega",
    "product_category_name_english":    "pre-entrega",
    "product_length_cm":                "pre-entrega",
    "product_height_cm":                "pre-entrega",
    "product_width_cm":                 "pre-entrega",
    "order_status":                     "pre-entrega",
    "order_purchase_timestamp":         "pre-entrega",
    # [pos-entrega] â PROIBIDO NO MODELO ML
    "order_delivered_customer_date":    "pos-entrega | PROIBIDO NO MODELO",
    "order_delivered_carrier_date":     "pos-entrega | PROIBIDO NO MODELO",
    "actual_delay_days":                "pos-entrega | PROIBIDO NO MODELO | derivada",
    "review_score":                     "pos-entrega | PROIBIDO NO MODELO",
    # [target]
    "bad_review":                       "target | 1=insatisfacao (1-2 estrelas), 0=ok",
}

In [22]:
# Celula 20: imprimir tabela de tagging e verificar colunas ausentes
print("=== COLUMN TAGS â CONTRATO DA TABELA GOLD ===\n")
missing_cols = []
for col, tag in COLUMN_TAGS.items():
    status = "[OK]" if col in gold_features.columns else "[AUSENTE]"
    if status == "[AUSENTE]":
        missing_cols.append(col)
    print(f"  {status} {col:45s} | {tag}")

if missing_cols:
    print(f"\nAVISO: Colunas ausentes em gold_features: {missing_cols}")
else:
    print("\nTodas as colunas do COLUMN_TAGS estao presentes em gold_features.")

# Renomear gold_features para gold_tagged (nome final)
gold_tagged = gold_features.copy()
print(f"\nTabela gold_tagged: {gold_tagged.shape}")
print(f"Colunas ({len(gold_tagged.columns)}): {sorted(gold_tagged.columns.tolist())}")

=== COLUMN TAGS â CONTRATO DA TABELA GOLD ===

  [OK] order_id                                      | pre-entrega | identificador
  [OK] order_approved_at                             | pre-entrega | ancora temporal
  [OK] order_estimated_delivery_date                 | pre-entrega
  [OK] estimated_days                                | pre-entrega | derivada
  [OK] payment_value                                 | pre-entrega | soma de itens
  [OK] freight_value                                 | pre-entrega
  [OK] freight_ratio                                 | pre-entrega | derivada
  [OK] total_payment_value                           | pre-entrega | soma de pagamentos
  [OK] product_weight_g                              | pre-entrega
  [OK] product_volume_cm3                            | pre-entrega | derivada
  [OK] product_photos_qty                            | pre-entrega
  [OK] seller_customer_distance_km                   | pre-entrega | derivada | Haversine km
  [OK] seller_id 

Colunas (38): ['actual_delay_days', 'bad_review', 'customer_city', 'customer_id', 'customer_lat', 'customer_lng', 'customer_state', 'customer_unique_id', 'customer_zip_code_prefix', 'estimated_days', 'freight_ratio', 'freight_value', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_id', 'order_purchase_timestamp', 'order_status', 'payment_value', 'product_category_name', 'product_category_name_english', 'product_height_cm', 'product_id', 'product_length_cm', 'product_photos_qty', 'product_volume_cm3', 'product_weight_g', 'product_width_cm', 'review_score', 'seller_city', 'seller_customer_distance_km', 'seller_id', 'seller_lat', 'seller_lng', 'seller_state', 'seller_zip_code_prefix', 'total_payment_value']


## Secao 9: Distribuicao do Target

In [23]:
# Celula 21: distribuicao do target â documentar para scale_pos_weight do XGBoost
bad_rate = gold_tagged["bad_review"].mean()
print(f"bad_review=1 (insatisfacao): {gold_tagged['bad_review'].sum()} pedidos ({bad_rate:.1%})")
print(f"bad_review=0 (ok): {(gold_tagged['bad_review']==0).sum()} pedidos ({1-bad_rate:.1%})")

# Validacoes finais da tabela gold_tagged
assert gold_tagged["order_id"].is_unique, "order_id deve ser unico em gold_tagged"
assert gold_tagged["bad_review"].isin([0, 1]).all(), "bad_review deve ser 0 ou 1"
assert gold_tagged["bad_review"].isna().sum() == 0, "bad_review nao pode ter NaN"
assert "estimated_days" in gold_tagged.columns, "estimated_days nao encontrado"
assert "freight_ratio" in gold_tagged.columns, "freight_ratio nao encontrado"
assert "product_volume_cm3" in gold_tagged.columns, "product_volume_cm3 nao encontrado"
assert "seller_customer_distance_km" in gold_tagged.columns, "seller_customer_distance_km nao encontrado"
assert "actual_delay_days" in gold_tagged.columns, "actual_delay_days nao encontrado"

dist_max = gold_tagged["seller_customer_distance_km"].max()
assert dist_max > 100, f"seller_customer_distance_km max={dist_max} parece em graus"

print(f"\nscale_pos_weight para XGBoost (se usar): {(1 - bad_rate) / bad_rate:.2f}")
print(f"\nTodos os asserts da tabela gold_tagged passaram com sucesso.")
print(f"gold_tagged esta pronto para export no Plan 03.")

bad_review=1 (insatisfacao): 13521 pedidos (13.9%)
bad_review=0 (ok): 83935 pedidos (86.1%)

scale_pos_weight para XGBoost (se usar): 6.21

Todos os asserts da tabela gold_tagged passaram com sucesso.
gold_tagged esta pronto para export no Plan 03.


## Secao 10: Checklist de Qualidade de Dados

In [24]:
# Celula 22: === CHECKLIST DE QUALIDADE DE DADOS ===
print("=" * 60)
print("CHECKLIST DE QUALIDADE  TABELA GOLD")
print("=" * 60)

# --- 1. Nulos por coluna ---
print("\n[1] NULOS POR COLUNA")
null_counts = gold_tagged.isnull().sum()
null_pct = (null_counts / len(gold_tagged) * 100).round(2)
null_report = pd.DataFrame({"nulos": null_counts, "pct": null_pct})
null_report = null_report[null_report["nulos"] > 0].sort_values("nulos", ascending=False)
if null_report.empty:
    print("  STATUS: OK  sem nulos")
else:
    print(null_report.to_string())

# Colunas criticas nao podem ter nulos
CRITICAS = ["order_id", "order_approved_at", "review_score", "bad_review",
            "seller_id", "customer_id", "seller_state", "customer_state"]
for col in CRITICAS:
    if col in gold_tagged.columns:
        n = gold_tagged[col].isna().sum()
        status = "OK" if n == 0 else f"PROBLEMA: {n} nulos"
        print(f"  {col}: {status}")

# --- 2. Duplicatas ---
print("\n[2] DUPLICATAS")
dup_order = gold_tagged["order_id"].duplicated().sum()
print(f"  order_id duplicados: {dup_order}  STATUS: {'OK' if dup_order == 0 else 'PROBLEMA'}")

# --- 3. CEPs invalidos ---
print("\n[3] CEPs INVALIDOS (formato esperado: string de 5 digitos)")
# CEP invalido = nao numerico ou diferente de 5 chars
def check_cep_valid(series):
    return series.astype(str).str.match(r"^\d{5}$")

seller_cep_ok = check_cep_valid(gold_tagged["seller_zip_code_prefix"]).sum()
customer_cep_ok = check_cep_valid(gold_tagged["customer_zip_code_prefix"]).sum()
total = len(gold_tagged)
print(f"  seller_zip validos: {seller_cep_ok}/{total} ({seller_cep_ok/total:.1%})")
print(f"  customer_zip validos: {customer_cep_ok}/{total} ({customer_cep_ok/total:.1%})")

seller_sem_geo = gold_tagged["seller_lat"].isna().sum()
customer_sem_geo = gold_tagged["customer_lat"].isna().sum()
print(f"  Sellers sem lat/lon (CEP nao encontrado no geo): {seller_sem_geo} ({seller_sem_geo/total:.1%})")
print(f"  Customers sem lat/lon (CEP nao encontrado no geo): {customer_sem_geo} ({customer_sem_geo/total:.1%})")

# --- 4. Datas inconsistentes ---
print("\n[4] DATAS INCONSISTENTES")
# order_approved_at deve existir (ja filtrado)
approved_nulos = gold_tagged["order_approved_at"].isna().sum()
print(f"  order_approved_at nulos: {approved_nulos}  STATUS: {'OK' if approved_nulos == 0 else 'PROBLEMA'}")

# estimated_delivery deve ser posterior a approved_at
bad_dates = (gold_tagged["order_estimated_delivery_date"] < gold_tagged["order_approved_at"]).sum()
print(f"  estimated_delivery < approved_at: {bad_dates} pedidos  STATUS: {'OK' if bad_dates == 0 else f'AVISO: {bad_dates}'}")

# estimated_days negativos
neg_days = (gold_tagged["estimated_days"] < 0).sum()
print(f"  estimated_days negativos: {neg_days}  STATUS: {'OK' if neg_days == 0 else f'AVISO: {neg_days}'}")

# Data range da ancora temporal
print(f"\n  Range order_approved_at: {gold_tagged['order_approved_at'].min().date()} a {gold_tagged['order_approved_at'].max().date()}")
print(f"  AVISO: dataset Olist termina abruptamente em Set 2018  se analise de tendencia, excluir Set 2018")

# --- 5. Distribuicao do target ---
print("\n[5] DISTRIBUICAO DO TARGET")
bad_rate = gold_tagged["bad_review"].mean()
print(f"  bad_review=1: {gold_tagged['bad_review'].sum()} ({bad_rate:.1%})")
print(f"  bad_review=0: {(gold_tagged['bad_review']==0).sum()} ({1-bad_rate:.1%})")
print(f"  scale_pos_weight sugerido para XGBoost: {(gold_tagged['bad_review']==0).sum() / gold_tagged['bad_review'].sum():.1f}")

# --- 6. Outliers de frete ---
print("\n[6] OUTLIERS DE FRETE")
frete_std = gold_tagged["freight_value"].std()
frete_mean = gold_tagged["freight_value"].mean()
outliers_frete = (gold_tagged["freight_value"] > frete_mean + 3*frete_std).sum()
print(f"  Pedidos com frete > media+3std: {outliers_frete} ({outliers_frete/total:.1%})")
print(f"  STATUS: incluidos no gold mas flagados  nao remover (decisao Phase 1)")

print("\n" + "=" * 60)
print("CHECKLIST CONCLUIDO")
print("=" * 60)


CHECKLIST DE QUALIDADE  TABELA GOLD

[1] NULOS POR COLUNA
                               nulos   pct
order_delivered_customer_date   1646  1.69
actual_delay_days               1646  1.69
product_category_name_english   1412  1.45
product_category_name           1393  1.43
product_photos_qty              1393  1.43
order_delivered_carrier_date     608  0.62
seller_customer_distance_km      490  0.50
customer_lng                     272  0.28
customer_lat                     272  0.28
seller_lng                       219  0.22
seller_lat                       219  0.22
product_length_cm                 19  0.02
product_weight_g                  19  0.02
product_volume_cm3                19  0.02
product_height_cm                 19  0.02
product_width_cm                  19  0.02
freight_ratio                      4  0.00
seller_id                          3  0.00
seller_city                        3  0.00
seller_zip_code_prefix             3  0.00
product_id                         3  

  order_id: OK
  order_approved_at: OK
  review_score: OK
  bad_review: OK
  seller_id: PROBLEMA: 3 nulos
  customer_id: OK
  seller_state: PROBLEMA: 3 nulos
  customer_state: OK

[2] DUPLICATAS
  order_id duplicados: 0  STATUS: OK

[3] CEPs INVALIDOS (formato esperado: string de 5 digitos)


  seller_zip validos: 97453/97456 (100.0%)
  customer_zip validos: 97456/97456 (100.0%)
  Sellers sem lat/lon (CEP nao encontrado no geo): 219 (0.2%)
  Customers sem lat/lon (CEP nao encontrado no geo): 272 (0.3%)

[4] DATAS INCONSISTENTES
  order_approved_at nulos: 0  STATUS: OK
  estimated_delivery < approved_at: 7 pedidos  STATUS: AVISO: 7
  estimated_days negativos: 7  STATUS: AVISO: 7

  Range order_approved_at: 2016-09-15 a 2018-09-03
  AVISO: dataset Olist termina abruptamente em Set 2018  se analise de tendencia, excluir Set 2018

[5] DISTRIBUICAO DO TARGET
  bad_review=1: 13521 (13.9%)
  bad_review=0: 83935 (86.1%)
  scale_pos_weight sugerido para XGBoost: 6.2

[6] OUTLIERS DE FRETE
  Pedidos com frete > media+3std: 1561 (1.6%)
  STATUS: incluidos no gold mas flagados  nao remover (decisao Phase 1)

CHECKLIST CONCLUIDO


## Secao 11: Export Gold Parquet

## 6. Export para Tabela Gold

**Formato parquet:** Mais eficiente que CSV para leitura em pandas (colunas tipadas, compressao, metadados). Qualquer notebook downstream faz `pd.read_parquet(ROOT / "data" / "gold" / "olist_gold.parquet")` sem joins adicionais.
**Imutabilidade:** Uma vez exportado, `olist_gold.parquet` e o contrato imutavel de dados — nao modificar sem documentar a mudanca e re-executar todos os notebooks downstream.
**Round-trip verificado:** Carregar o parquet imediatamente apos salvar confirma que shapes e dtypes estao corretos antes de qualquer uso downstream.

In [25]:
# Celula 23: selecionar colunas finais e exportar
# Manter todas as colunas (pre-entrega + pos-entrega + target)  tags no COLUMN_TAGS definem uso
# Nao filtrar colunas aqui  o contrato e a tabela completa

DATA_GOLD_EXPORT = PROJECT_ROOT / "data" / "gold"
DATA_GOLD_EXPORT.mkdir(parents=True, exist_ok=True)

gold_tagged.to_parquet(DATA_GOLD_EXPORT / "olist_gold.parquet", index=False)
print(f"Exportado: {DATA_GOLD_EXPORT / 'olist_gold.parquet'}")
print(f"Shape: {gold_tagged.shape}")


Exportado: C:\Users\Wey\Desktop\Alpha\Projetos\python\data\gold\olist_gold.parquet
Shape: (97456, 38)


In [26]:
# Celula 24: verificacao de round-trip (carregar e confirmar que bate)
gold_check = pd.read_parquet(DATA_GOLD_EXPORT / "olist_gold.parquet")
assert gold_check.shape == gold_tagged.shape, "Round-trip falhou: shapes divergem"
assert "bad_review" in gold_check.columns, "Coluna target bad_review ausente apos round-trip"
assert "seller_customer_distance_km" in gold_check.columns, "Coluna de distancia ausente"
assert gold_check["order_id"].is_unique, "order_id duplicado apos round-trip"
print(f"Round-trip OK  {gold_check.shape[0]} linhas, {gold_check.shape[1]} colunas")
print(f"Dtypes:\n{gold_check.dtypes.to_string()}")


Round-trip OK  97456 linhas, 38 colunas
Dtypes:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
freight_value                           float64
payment_value                           float64
seller_id                                   str
product_id                                  str
review_score                            float64
total_payment_value                     float64
customer_unique_id                          str
customer_zip_code_prefix                    str
customer_city                               str
customer_state                              str
seller_zip_code_prefix                      str
seller_city                            

## Resumo — Tabela Gold

| Metrica | Valor |
|---------|-------|
| Total de pedidos | 97.456 linhas |
| Colunas totais | 38 colunas |
| bad_review = 1 | 13.521 pedidos (13.9%) |
| Nulos criticos | order_id, bad_review, customer_id: 0 nulos. seller_id: 3 nulos (0.003%) — aceito |
| Distancia min/max (km) | 0 — 8.677 km |
| Cobertura geo sellers | 99.8% (219 sem lat/lon) |
| Cobertura geo customers | 99.7% (272 sem lat/lon) |

**Proxima etapa:** `FASE3-P3-eda.ipynb` consome este parquet para analise exploratoria.

*Nota: scale_pos_weight XGBoost = 6.21 (negativo/positivo = 83.935/13.521)*